# SesliKitap — Kaggle / Google Colab VoxCPM2 TTS Sunucusu (`voxcpm2_alt`)

Bu defter, ücretsiz T4 GPU (16 GB VRAM) üzerinde aynı `tts_server` paketini çalıştırır ve `cloudflared` tüneliyle geçici bir HTTPS uç noktası üretir.

In [ ]:
import os

os.environ['TTS_AUTH_TOKEN'] = 'dev-tts-secret-token-change-me-32chars'
os.environ['TTS_ENGINE'] = 'voxcpm2'
os.environ['TTS_MODEL_ID'] = 'openbmb/VoxCPM2'
os.environ['TTS_VOICES_DIR'] = '/tmp/seslikitap_voices'
os.environ['HF_HOME'] = '/tmp/hf_cache'

In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1 wget
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!pip install -q voxcpm fastapi "uvicorn[standard]" pydantic pydantic-settings soundfile numpy huggingface-hub

In [ ]:
!if [ ! -d SesliKitap ]; then git clone https://github.com/Muhammed-KRM/SesliKitap.git; fi
%cd SesliKitap/gpu

In [ ]:
import re
import subprocess
import threading
import time

import uvicorn
from tts_server.local_main import create_local_app

app = create_local_app()
threading.Thread(target=lambda: uvicorn.run(app, host='127.0.0.1', port=8009), daemon=True).start()
time.sleep(3)

proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8009', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
for line in proc.stdout:
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        print('============================================================')
        print('TTS_ALT_URL=' + m.group(0))
        print('TTS_ALT_HEALTH_URL=' + m.group(0) + '/v1/health')
        print('============================================================')
